# RQ1 sampling for open coding

Draws the samples the coders work on. Run `load_dataset.ipynb` first: its last cell saves
`rq1_frame.parquet`, which this notebook reads.

Two phases:

1. **Codebook rounds.** Small samples (default 50). Two coders code each round independently,
   resolve conflicts, update the codebook. Repeat with new pairs until a round adds no new codes.
2. **Measurement sample.** With the codebook frozen, code a fresh sample sized for a
   95% confidence level and 5% margin of error, stratified by transition type.

Nothing is ever drawn twice. Every draw is saved to `coding_rounds/`, and later draws skip any
pair already in a saved file.

**Coders work blind.** The coding sheet does not show the security label (`llm_transition`),
so it can't influence the codes. The label sits in a separate key file and is only joined back
after coding.

In [9]:
import math
from pathlib import Path

import pandas as pd
from IPython.display import display

FRAME_PATH = Path("rq1_frame.parquet")
CODING_DIR = Path("coding_rounds")
CODING_DIR.mkdir(exist_ok=True)

SEED_BASE = 2026   # change only if you restart sampling from scratch

frame = pd.read_parquet(FRAME_PATH)

frame["pair_id"] = (frame["skill_slug"] + "@" +
                    frame["prev_version"].astype(str) + "->" +
                    frame["version"].astype(str))
assert frame["pair_id"].is_unique, "pair_id is not unique, check the frame"

# Transition groups used for stratifying
GROUP = {"introduced": "introduced", "fixed": "fixed",
         "stayed clean": "no change", "stayed flagged": "no change"}
frame["stratum"] = frame["llm_transition"].map(GROUP)
frame = frame[frame["stratum"].notna()].copy()

print(f"{len(frame):,} pairs in the sampling frame, {frame['skill_slug'].nunique():,} skills\n")
display(pd.crosstab(frame["llm_transition"], frame["changelog_source"],
                    margins=True, margins_name="total"))

93,094 pairs in the sampling frame, 21,723 skills



changelog_source,auto,user,total
llm_transition,,,
fixed,2617,8164,10781
introduced,3301,11068,14369
stayed clean,10423,28053,38476
stayed flagged,8253,21215,29468
total,24594,68500,93094


In [10]:
# Columns the coders see, and columns kept back in the key file
SHEET_COLS = ["pair_id", "skill_slug", "prev_version", "version",
              "changelog_source", "changelog_text"]
KEY_COLS = ["pair_id", "llm_transition", "stratum", "pipeline_transition",
            "pipeline_changed", "model_changed", "changelog_source"]
CODER_COLS = ["coder1_code", "coder2_code", "agreed_code", "notes"]

def already_drawn():
    # Every pair_id in any saved key file, from any phase.
    files = list(CODING_DIR.glob("*_key.csv"))
    if not files:
        return set()
    return set(pd.concat(pd.read_csv(f, usecols=["pair_id"]) for f in files)["pair_id"])

def remaining_pool():
    return frame[~frame["pair_id"].isin(already_drawn())]

def save_draw(sample, name):
    sheet_path = CODING_DIR / f"{name}_sheet.csv"
    key_path = CODING_DIR / f"{name}_key.csv"
    if sheet_path.exists() or key_path.exists():
        raise FileExistsError(f"{name} already exists. Use a new name/round number; "
                              f"never overwrite a drawn sample.")
    # shuffle row order so coders don't see pairs grouped by stratum
    sample = sample.sample(frac=1, random_state=SEED_BASE)
    sheet = sample[[c for c in SHEET_COLS if c in sample.columns]].copy()
    for c in CODER_COLS:
        sheet[c] = ""
    sheet.to_csv(sheet_path, index=False)
    sample[[c for c in KEY_COLS if c in sample.columns]].to_csv(key_path, index=False)
    print(f"saved {len(sample)} pairs")
    print(f"  coding sheet (give to coders): {sheet_path}")
    print(f"  key (keep back until coding is done): {key_path}")
    print(f"  {len(remaining_pool()):,} pairs left in the pool")
    print("\nby stratum:")
    display(sample["stratum"].value_counts().rename("pairs").to_frame())

## Phase 1: codebook rounds

Each round draws `n` new pairs. With `balanced=True` the round is split evenly across
introduced / fixed / no change, so the coders see all kinds of transition while building
the codebook. Set it to `False` for a plain random draw.

Change `ROUND` each time. The cell refuses to overwrite a round that already exists.

In [11]:
ROUND = 1
N_PER_ROUND = 50
BALANCED = True

def draw_round(round_no, n=N_PER_ROUND, balanced=BALANCED):
    name = f"phase1_round{round_no:02d}"
    if (CODING_DIR / f"{name}_key.csv").exists():
        print(f"{name} was already drawn, nothing new drawn. Raise ROUND for a new round.")
        return pd.read_csv(CODING_DIR / f"{name}_sheet.csv")
    pool = remaining_pool()
    seed = SEED_BASE + round_no
    if balanced:
        groups = sorted(pool["stratum"].unique())
        per = {g: n // len(groups) + (1 if i < n % len(groups) else 0)
               for i, g in enumerate(groups)}
        parts = [g.sample(n=min(per[k], len(g)), random_state=seed)
                 for k, g in pool.groupby("stratum")]
        sample = pd.concat(parts)
    else:
        sample = pool.sample(n=n, random_state=seed)
    save_draw(sample, name)
    return sample

round_sample = draw_round(ROUND)

saved 50 pairs
  coding sheet (give to coders): coding_rounds\phase1_round01_sheet.csv
  key (keep back until coding is done): coding_rounds\phase1_round01_key.csv
  93,044 pairs left in the pool

by stratum:


,pairs
stratum,
introduced,17
fixed,17
no change,16


Diffs

In [12]:
import difflib
import gzip
import html
import io
import json
import re
import time
import zipfile

import requests
from tqdm.auto import tqdm

API = "https://clawhub.ai/api/v1"
FILES_CACHE = CODING_DIR / "_version_files"   # cached files of each downloaded version
MAX_FILE_BYTES = 1_000_000                    # bigger files are listed as changed but not diffed
SHOW_LINES_PER_FILE = 300                     # diff lines shown per file in the HTML view
session = requests.Session()


def _safe(value):
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(value))


def fetch_zip(owner, slug, version, tries=5):
    status = "no response"
    for attempt in range(tries):
        try:
            r = session.get(f"{API}/download", timeout=60,
                            params={"slug": slug, "ownerHandle": owner, "version": version})
        except requests.RequestException:
            r = None
        if r is not None:
            status = r.status_code
            if r.status_code == 200:
                return r.content, "ok"
            if r.status_code == 404:
                return None, "not found"
        wait = 2 ** attempt
        if r is not None and r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 10 * (attempt + 1)))
        time.sleep(wait)
    return None, f"failed ({status})"


def version_files(skill_slug, version):
    """One version's files: ({path: text}, {path: crc of non-text files}, status). Cached."""
    owner, slug = skill_slug.split("/", 1)
    path = FILES_CACHE / _safe(owner) / f"{_safe(slug)}__{_safe(version)}.json.gz"
    if path.exists():
        with gzip.open(path, "rt", encoding="utf-8") as f:
            d = json.load(f)
        return d["text"], d["other"], "ok"
    blob, status = fetch_zip(owner, slug, version)
    if blob is None:
        return None, None, status
    text, other = {}, {}
    try:
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                if info.file_size <= MAX_FILE_BYTES:
                    try:
                        text[info.filename] = zf.read(info).decode("utf-8")
                        continue
                    except (UnicodeDecodeError, RuntimeError):
                        pass
                other[info.filename] = info.CRC
    except zipfile.BadZipFile:
        return None, None, "bad zip"
    # drop a top folder shared by every file (e.g. "my-skill/"), so paths match across versions
    names = list(text) + list(other)
    tops = {n.split("/", 1)[0] for n in names}
    if names and len(tops) == 1 and all("/" in n for n in names):
        cut = len(tops.pop()) + 1
        text = {n[cut:]: t for n, t in text.items()}
        other = {n[cut:]: c for n, c in other.items()}
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump({"text": text, "other": other}, f)
    return text, other, "ok"


def pair_diff(prev, curr):
    """Per changed file: dict(path, kind, added, removed, lines). SKILL.md first."""
    (pt, po), (ct, co) = prev, curr
    names = set(pt) | set(po) | set(ct) | set(co)
    IGNORE = {"skill-card.md", "_meta.json"}
    names = {n for n in names if n.split("/")[-1].lower() not in IGNORE}
    order = sorted(names, key=lambda n: (n.split("/")[-1].upper() != "SKILL.MD", n.lower()))
    out = []
    for name in order:
        in_prev, in_curr = name in pt or name in po, name in ct or name in co
        kind = "added" if not in_prev else "deleted" if not in_curr else "modified"
        if name in po or name in co:             # binary or too big on at least one side
            if po.get(name) is not None and po.get(name) == co.get(name):
                continue
            out.append({"path": name, "kind": kind + ", binary or too big, not shown",
                        "added": 0, "removed": 0, "lines": []})
            continue
        before, after = pt.get(name, "").splitlines(), ct.get(name, "").splitlines()
        if in_prev and in_curr and before == after:
            continue
        lines = list(difflib.unified_diff(before, after, f"a/{name}", f"b/{name}", lineterm=""))
        body = [l for l in lines[2:] if not l.startswith("@@")]
        out.append({"path": name, "kind": kind,
                    "added": sum(l.startswith("+") for l in body),
                    "removed": sum(l.startswith("-") for l in body),
                    "lines": lines[2:]})
    return out


CSS = """
body{font:15px/1.45 system-ui,Segoe UI,Arial,sans-serif;margin:0;background:#f6f5f2;color:#1e2a44}
main{max-width:1100px;margin:0 auto;padding:24px 16px}
h1{font-size:22px} .pair{background:#fff;border:1px solid #ddd8cc;border-radius:10px;padding:16px;margin:28px 0}
.pair h2{font-size:17px;margin:0 0 4px} .meta{color:#5b6474;font-size:13px}
.label{font-weight:600;margin:14px 0 6px} .cl{white-space:pre-wrap;background:#fbf8ef;border-left:4px solid #e0b450;padding:10px 12px;border-radius:4px}
details{border:1px solid #e5e1d8;border-radius:6px;margin:6px 0} summary{cursor:pointer;padding:6px 10px;font:13px ui-monospace,Consolas,monospace;background:#f3f1ec}
pre{margin:0;padding:6px 0;font:12.5px/1.4 ui-monospace,Consolas,monospace;white-space:pre-wrap;word-break:break-all}
pre span{display:block;padding:0 10px} .a{background:#e6f6ec} .r{background:#fbe9e9} .h{color:#7a8496;background:#f3f1ec}
.more{color:#5b6474;font-style:italic} .add{color:#1f7a5a} .del{color:#b42318}
table{border-collapse:collapse;font-size:13px} td,th{padding:3px 8px;border-bottom:1px solid #e5e1d8;text-align:left}
a{color:#2f5bd3}
"""


def _pair_html(n, row, files, status):
    esc = html.escape
    parts = [f'<section class="pair" id="p{n}"><h2>#{n} &nbsp;{esc(row.pair_id)}</h2>',
             f'<div class="meta">sheet row {n} (Excel row {n + 1})</div>',
             '<div class="label">Changelog (what the author says changed)</div>',
             f'<div class="cl">{esc(str(row.changelog_text))}</div>',
             '<div class="label">Diff (what actually changed)</div>']
    if files is None:
        parts.append(f'<p class="more">Diff not available: {esc(status)}</p>')
    elif not files:
        parts.append('<p class="more">No file changes between these two versions.</p>')
    for f in files or []:
        lines = f["lines"]
        shown = lines[:SHOW_LINES_PER_FILE]
        rows = []
        for l in shown:
            cls = "h" if l.startswith("@@") else "a" if l.startswith("+") else "r" if l.startswith("-") else ""
            rows.append(f'<span class="{cls}">{esc(l) or "&nbsp;"}</span>')
        if len(lines) > len(shown):
            rows.append(f'<span class="more">... {len(lines) - len(shown):,} more lines, '
                        f'see the full .diff file</span>')
        opened = " open" if len(files) <= 6 else ""
        parts.append(f'<details{opened}><summary>{esc(f["path"])} &nbsp;({esc(f["kind"])}) '
                     + ("" if not f["lines"] else
                        f'<span class="add">+{f["added"]}</span> <span class="del">-{f["removed"]}</span>')
                     + '</summary>'
                     + (f'<pre>{"".join(rows)}</pre>' if rows else "") + '</details>')
    parts.append("</section>")
    return "".join(parts)


def _read_sheet(name):
    csv_path, xlsx_path = CODING_DIR / f"{name}_sheet.csv", CODING_DIR / f"{name}_sheet.xlsx"
    if csv_path.exists():
        return pd.read_csv(csv_path, encoding="utf-8-sig", dtype=str).fillna("")
    return pd.read_excel(xlsx_path, dtype=str).fillna("")


def attach_diffs(name, pause=0.3):
    """Writes <name>_diffs.html (coding view), <name>_diffstats.csv and <name>_diffs/NNN.diff.
    Never touches the coding sheet itself."""
    sheet = _read_sheet(name)
    full_dir = CODING_DIR / f"{name}_diffs"
    full_dir.mkdir(exist_ok=True)
    stats, cards = [], []
    for n, row in enumerate(tqdm(list(sheet.itertuples(index=False)), desc=name), start=1):
        pt, po, s1 = version_files(row.skill_slug, row.prev_version)
        time.sleep(pause)
        ct, co, s2 = version_files(row.skill_slug, row.version)
        time.sleep(pause)
        if pt is None or ct is None:
            files, status = None, f"previous version: {s1}, this version: {s2}"
        else:
            files, status = pair_diff((pt, po), (ct, co)), "ok"
            text = "\n".join(f'--- a/{f["path"]}\n+++ b/{f["path"]}\n' + "\n".join(f["lines"])
                             if f["lines"] else f'# {f["path"]}: {f["kind"]}' for f in files)
            (full_dir / f"{n:03d}.diff").write_text(text, encoding="utf-8")
        cards.append(_pair_html(n, row, files, status))
        stats.append({"row": n, "pair_id": row.pair_id, "diff_status": status,
                      "files_changed": None if files is None else len(files),
                      "lines_added": None if files is None else sum(f["added"] for f in files),
                      "lines_removed": None if files is None else sum(f["removed"] for f in files),
                      "changed_files": "" if files is None else "; ".join(f["path"] for f in files)})
    stats = pd.DataFrame(stats)
    stats.to_csv(CODING_DIR / f"{name}_diffstats.csv", index=False, encoding="utf-8-sig")
    index = "".join(
        f'<tr><td><a href="#p{r.row}">#{r.row}</a></td><td>{html.escape(r.pair_id)}</td>'
        f'<td>{"" if pd.isna(r.files_changed) else int(r.files_changed)}</td>'
        f'<td class="add">{"" if pd.isna(r.lines_added) else "+" + str(int(r.lines_added))}</td>'
        f'<td class="del">{"" if pd.isna(r.lines_removed) else "-" + str(int(r.lines_removed))}</td></tr>'
        for r in stats.itertuples())
    page = (f'<!doctype html><html><head><meta charset="utf-8"><title>{html.escape(name)} diffs</title>'
            f'<style>{CSS}</style></head><body><main><h1>{html.escape(name)}: changelog + diff</h1>'
            '<p class="meta">Same order as the coding sheet. Green lines were added, red lines removed. '
            'Security labels are not shown here, so coding stays blind.</p>'
            '<table><tr><th>#</th><th>pair</th><th>files</th><th>added</th><th>removed</th></tr>'
            f'{index}</table>{"".join(cards)}</main></body></html>')
    out = CODING_DIR / f"{name}_diffs.html"
    out.write_text(page, encoding="utf-8")
    ok = (stats["diff_status"] == "ok").sum()
    print(f"{ok} of {len(stats)} pairs have a diff")
    print(f"  open in a browser next to the sheet: {out}")
    print(f"  stats: {CODING_DIR / f'{name}_diffstats.csv'}   full diffs: {full_dir}")
    if ok < len(stats):
        display(stats[stats["diff_status"] != "ok"][["row", "pair_id", "diff_status"]])
    return stats


# ===================== CODE CELL 2: build the view for a round =====================
# Run after each draw. Works for round 1 too, even though it is already coded.
diff_stats = attach_diffs(f"phase1_round{ROUND:02d}")
# for the measurement sample later:  attach_diffs("phase2")


phase1_round01:   0%|          | 0/50 [00:00<?, ?it/s]

50 of 50 pairs have a diff
  open in a browser next to the sheet: coding_rounds\phase1_round01_diffs.html
  stats: coding_rounds\phase1_round01_diffstats.csv   full diffs: coding_rounds\phase1_round01_diffs


## After each round: agreement and new codes

Fill in `coder1_code` and `coder2_code` independently, then `agreed_code` after resolving
conflicts. One code per pair. If a pair needs several, separate them with `;` and the
first one is treated as the primary code for the kappa.

The table at the end tracks saturation: when a round adds no new codes, Phase 1 is done.

In [ ]:
def cohens_kappa(a, b):
    a, b = pd.Series(a).reset_index(drop=True), pd.Series(b).reset_index(drop=True)
    po = (a == b).mean()
    cats = set(a) | set(b)
    pe = sum((a == c).mean() * (b == c).mean() for c in cats)
    return float("nan") if pe == 1 else (po - pe) / (1 - pe)

def primary(s):
    return s.fillna("").astype(str).str.split(";").str[0].str.strip().str.lower()

def round_report():
    rows, seen = [], set()
    for sheet_path in sorted(CODING_DIR.glob("phase1_round*_sheet.csv")):
        s = pd.read_csv(sheet_path).fillna("")
        done = s[(s["coder1_code"] != "") & (s["coder2_code"] != "")]
        c1, c2 = primary(done["coder1_code"]), primary(done["coder2_code"])
        agreed = set()
        for v in s["agreed_code"]:
            agreed |= {c.strip().lower() for c in str(v).split(";") if c.strip()}
        new = agreed - seen
        seen |= agreed
        rows.append({
            "round": sheet_path.stem.replace("_sheet", ""),
            "pairs": len(s),
            "coded_by_both": len(done),
            "raw_agreement": (c1 == c2).mean() if len(done) else None,
            "kappa": cohens_kappa(c1, c2) if len(done) else None,
            "codes_so_far": len(seen),
            "new_codes": len(new),
            "which_new": ", ".join(sorted(new)),
        })
    out = pd.DataFrame(rows)
    if len(out):
        display(out.style.format({"raw_agreement": "{:.1%}", "kappa": "{:.2f}"}, na_rep="-"))
    else:
        print("no rounds drawn yet")
    return out

report = round_report()

## Phase 2: measurement sample

Only run this once the codebook is frozen.

Sample size per stratum uses the standard formula with a finite population correction:
n0 = z² p(1-p) / e², n = n0 / (1 + (n0 - 1) / N), with z = 1.96, p = 0.5, e = 0.05.

- **Option A** (`PER_STRATUM = True`): a full 95/5 sample in each stratum. About 380 per
  stratum, so roughly 1,150 pairs.
- **Option B** (`PER_STRATUM = False`): one 95/5 sample of about 385, split evenly across
  strata. Cheaper, but each stratum's margin of error is wider. The cell prints it.

Phase 1 pairs are excluded automatically, since they shaped the codebook.

In [ ]:
PER_STRATUM = False   # True = option A, False = option B
Z, P, E = 1.96, 0.5, 0.05

def sample_size(N, z=Z, p=P, e=E):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

def moe(n, N, z=Z, p=P):
    fpc = math.sqrt((N - n) / (N - 1)) if N > 1 else 0
    return z * math.sqrt(p * (1 - p) / n) * fpc

pool = remaining_pool()
groups = dict(tuple(pool.groupby("stratum")))

if PER_STRATUM:
    sizes = {k: sample_size(len(g)) for k, g in groups.items()}
else:
    total = sample_size(len(pool))
    sizes = {k: math.ceil(total / len(groups)) for k in groups}

plan = pd.DataFrame([
    {"stratum": k, "pool": len(g), "sample": min(sizes[k], len(g)),
     "margin_of_error": moe(min(sizes[k], len(g)), len(g))}
    for k, g in groups.items()
])
print("Option", "A (95/5 per stratum)" if PER_STRATUM else "B (one 95/5 sample, split evenly)")
display(plan.style.format({"margin_of_error": "{:.1%}"}))
print(f"total pairs to code: {int(plan['sample'].sum()):,}")

In [ ]:
# Draw it. Run only after checking the plan above.
DRAW_PHASE2 = False

if (CODING_DIR / "phase2_key.csv").exists():
    print("phase 2 was already drawn, nothing new drawn.")
elif DRAW_PHASE2:
    parts = [g.sample(n=min(sizes[k], len(g)), random_state=SEED_BASE + 999)
             for k, g in groups.items()]
    phase2 = pd.concat(parts)
    save_draw(phase2, "phase2")
else:
    print("set DRAW_PHASE2 = True to draw")

## After Phase 2 coding: join the codes back to the labels

Only now are the coded sheets joined to the key files, so the security label meets the code
for the first time here.

In [ ]:
def load_coded(name):
    sheet = pd.read_csv(CODING_DIR / f"{name}_sheet.csv").fillna("")
    key = pd.read_csv(CODING_DIR / f"{name}_key.csv")
    return sheet.merge(key, on="pair_id", how="left", suffixes=("", "_key"))

if (CODING_DIR / "phase2_sheet.csv").exists():
    coded = load_coded("phase2")
    done = coded[coded["agreed_code"] != ""]
    print(f"{len(done):,} of {len(coded):,} phase 2 pairs have an agreed code\n")
    if len(done):
        c1, c2 = primary(done["coder1_code"]), primary(done["coder2_code"])
        print(f"raw agreement {(c1 == c2).mean():.1%}, kappa {cohens_kappa(c1, c2):.2f}\n")
        codes = done.assign(code=done["agreed_code"].str.split(";")).explode("code", ignore_index=True)
        codes["code"] = codes["code"].str.strip().str.lower()
        counts = pd.crosstab(codes["code"], codes["stratum"], margins=True, margins_name="total")
        print("code x stratum (counts):")
        display(counts)
        shares = pd.crosstab(codes["code"], codes["stratum"], normalize="columns")
        print("code x stratum (share within each stratum):")
        display(shares.style.format("{:.1%}"))
else:
    print("no phase 2 sample yet")